# Revacc T4 Backend — Full Pipeline with Local ESMFold

Runs the reverse-vaccinology backend on a T4 GPU with **local ESMFold** (OpenFold-compiled). Connects to Vercel frontend via ngrok.

1. Clone repo + install deps
2. Build OpenFold (C++17 patched) for CUDA
3. Download ESMFold 3B weights
4. Verify local ESMFold runs on T4
5. Start FastAPI backend + ngrok tunnel
6. Run full 50-step pipeline

In [ ]:
#@title 1. Clone repo and install system dependencies
!apt-get update -qq && apt-get install -y -qq blast+ ncbi-blast+ > /dev/null 2>&1

import os, subprocess
REPO_URL = "https://github.com/umeshdahiya15/Revacc.git"
WORKDIR = "/content/Revacc"

if not os.path.exists(WORKDIR):
    get_ipython().system(f"git clone {REPO_URL} {WORKDIR}")
else:
    os.chdir(WORKDIR)
    get_ipython().system("git pull origin main")

os.chdir(WORKDIR)
print(f"Working directory: {os.getcwd()}")

In [ ]:
#@title 2. Set up Python 3.10 venv and install backend deps
!apt-get install -y -qq python3.10 python3.10-venv python3.10-dev > /dev/null 2>&1

VENV = "/content/esmfold-env"
!python3.10 -m venv {VENV}
!{VENV}/bin/pip install --upgrade pip setuptools wheel > /dev/null 2>&1
!{VENV}/bin/pip install -r backend/requirements.txt > /dev/null 2>&1

# Install torch with CUDA for T4
!{VENV}/bin/pip install torch torchvision --index-url https://download.pytorch.org/whl/cu121 > /dev/null 2>&1

print("Python venv ready:", 
      subprocess.check_output([f"{VENV}/bin/python", "--version"]).decode().strip())

In [ ]:
#@title 3. Build OpenFold with C++17 patch (required for PyTorch 2.x)
import subprocess, os

VENV = "/content/esmfold-env"
OPENFOLD_DIR = "/content/openfold"

if not os.path.exists(OPENFOLD_DIR):
    # Clone with --filter=blob:none to allow checking out older commits
    !git clone --filter=blob:none https://github.com/aqlaboratory/openfold.git {OPENFOLD_DIR}
    os.chdir(OPENFOLD_DIR)
    !git checkout 4b41059  # Last working commit with Python 3.10 support
else:
    os.chdir(OPENFOLD_DIR)

# CRITICAL: Patch C++14 to C++17 in setup.py for PyTorch 2.x compatibility
!sed -i 's/-std=c++14/-std=c++17/g' setup.py
!grep -n 'c++17' setup.py | head -3

# Also patch any C++ extension files
!find . -name '*.cpp' -exec sed -i 's/-std=c++14/-std=c++17/g' {} + 2>/dev/null || true
!find . -name 'CMakeLists.txt' -exec sed -i 's/c++14/c++17/g' {} + 2>/dev/null || true

print("OpenFold C++17 patch applied.")

In [ ]:
#@title 4. Install OpenFold dependencies + compile CUDA extensions
VENV = "/content/esmfold-env"
OPENFOLD_DIR = "/content/openfold"

# Install openfold requirements
!{VENV}/bin/pip install -r {OPENFOLD_DIR}/requirements.txt > /dev/null 2>&1

# Install older PyTorch Lightning (required by this commit)
!{VENV}/bin/pip install pytorch-lightning==1.9.5 > /dev/null 2>&1

# Build OpenFold with CUDA extensions
os.chdir(OPENFOLD_DIR)
!{VENV}/bin/python setup.py install 2>&1 | tail -20

print("OpenFold installation complete.")

In [ ]:
#@title 5. Download ESMFold 3B weights (2.7 GB)
import os

WEIGHTS_DIR = os.path.expanduser("~/.cache/torch/hub/checkpoints")
WEIGHTS_FILE = os.path.join(WEIGHTS_DIR, "esmfold_3B_v1.pt")

if not os.path.exists(WEIGHTS_FILE):
    os.makedirs(WEIGHTS_DIR, exist_ok=True)
    !wget -q --show-progress -O {WEIGHTS_FILE} \
      https://dl.fbaipublicfiles.com/fair-esm/models/esmfold_3B_v1.pt
else:
    size_gb = os.path.getsize(WEIGHTS_FILE) / (1024**3)
    print(f"Weights already downloaded: {size_gb:.1f} GB")

print(f"Weights path: {WEIGHTS_FILE}")

In [ ]:
#@title 6. Verify local ESMFold runs on T4 GPU
VENV = "/content/esmfold-env"
WEIGHTS_FILE = os.path.expanduser("~/.cache/torch/hub/checkpoints/esmfold_3B_v1.pt")

verify_script = '''
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A'}")

# Try loading ESMFold via OpenFold
try:
    from openfold.inference_speed import AlphaFold
    print("OpenFold AlphaFold class imported.")
except ImportError:
    try:
        from openfold.utils.model_config import model_config
        print("OpenFold model_config imported.")
    except ImportError:
        print("WARNING: OpenFold import failed. Using torch.hub fallback.")

# Direct torch.hub load (most reliable)
print("Loading ESMFold model...")
model = torch.hub.load("facebookresearch/esm:main", "esmfold_3B_v1")
model = model.eval().cuda()
print(f"ESMFold loaded on: {next(model.parameters()).device}")

# Quick test: fold a short sequence
test_seq = "MKFLILLFNILCLFPVLAADNHGVSLQGFNKENYEKFDKARLENGITYDSIMYSGRDFNE"
with torch.no_grad():
    output = model.infer_pdb(test_seq)
print(f"Test fold complete: {len(output)} chars PDB")
print("LOCAL ESMFOLD WORKING ON T4!")
'''

with open('/tmp/verify_esmfold.py', 'w') as f:
    f.write(verify_script)

!{VENV}/bin/python /tmp/verify_esmfold.py

In [ ]:
#@title 7. Patch backend to use local ESMFold on T4
import os

WORKDIR = "/content/Revacc"
VENV = "/content/esmfold-env"

# Create a local ESMFold wrapper that loads from torch.hub on GPU
local_esmfold_script = '''
"""Local ESMFold wrapper for T4 GPU backend."""
import os
import torch

_model = None

def _get_model():
    global _model
    if _model is None:
        _model = torch.hub.load("facebookresearch/esm:main", "esmfold_3B_v1")
        _model = _model.eval().cuda()
    return _model

def predict_local(sequence: str) -> str:
    """Fold a sequence using the local ESMFold model on GPU."""
    model = _get_model()
    with torch.no_grad():
        pdb_text = model.infer_pdb(sequence)
    return pdb_text
'''

wrapper_path = os.path.join(WORKDIR, "backend", "app", "tools", "esmfold_local.py")
with open(wrapper_path, 'w') as f:
    f.write(local_esmfold_script)

print(f"Created local ESMFold wrapper: {wrapper_path}")

In [ ]:
#@title 8. Patch runner_additions.py to try local ESMFold first
import re

WORKDIR = "/content/Revacc"
runner_path = os.path.join(WORKDIR, "backend", "app", "tools", "runner_additions.py")

with open(runner_path, 'r') as f:
    content = f.read()

# Patch the ESMFold section in run_11_2 to try local first
old_esmfold_block = '''
            try:
                coordinate_text = await esmfold.predict_pdb(mev_seq)
            except esmfold.ESMFoldError as exc:
                raise ToolUnavailableError(
                    tool_name="ESMFold",
                    reason=str(exc),
                    workaround=(
                        "Use a local CUDA-enabled ESMFold runtime for this MEV, "
                        "or configure SWISS-MODEL with a valid CoreAPI token."
                    ),
                ) from exc
            raw_input = {
                "sequence": mev_seq,
                "source": "real",
                "provider": "ESMFold",
                "method": "ESMFold public API",
                "modelFormat": "pdb",
                "coordinateText": coordinate_text,
            }'''

new_esmfold_block = '''
            # Try local ESMFold (T4 GPU) first, then API fallback
            coordinate_text = None
            method_label = "ESMFold public API"
            try:
                from . import esmfold_local
                import asyncio
                coordinate_text = await asyncio.to_thread(
                    esmfold_local.predict_local, mev_seq
                )
                method_label = "ESMFold local (T4 GPU)"
            except Exception:
                try:
                    coordinate_text = await esmfold.predict_pdb(mev_seq)
                except esmfold.ESMFoldError as exc:
                    raise ToolUnavailableError(
                        tool_name="ESMFold",
                        reason=str(exc),
                        workaround=(
                            "Use a local CUDA-enabled ESMFold runtime for this MEV, "
                            "or configure SWISS-MODEL with a valid CoreAPI token."
                        ),
                    ) from exc
            raw_input = {
                "sequence": mev_seq,
                "source": "real",
                "provider": "ESMFold",
                "method": method_label,
                "modelFormat": "pdb",
                "coordinateText": coordinate_text,
            }'''

if old_esmfold_block in content:
    content = content.replace(old_esmfold_block, new_esmfold_block)
    with open(runner_path, 'w') as f:
        f.write(content)
    print("Patched runner_additions.py: local ESMFold first, API fallback.")
else:
    print("Patch target not found - may already be patched or pattern changed.")

In [ ]:
#@title 9. Install ngrok and get tunnel token
!pip install pyngrok > /dev/null 2>&1

#@title Configure ngrok (set your authtoken)
NGROK_TOKEN = ""  #@param {type:"string"}

if NGROK_TOKEN:
    from pyngrok import ngrok
    ngrok.set_auth_token(NGROK_TOKEN)
    print(f"ngrok auth token set.")
else:
    print("WARNING: No ngrok token. Will use localhost only.")

In [ ]:
#@title 10. Start backend server + ngrok tunnel
import os, subprocess, time, signal

WORKDIR = "/content/Revacc"
VENV = "/content/esmfold-env"
BACKEND_PORT = 8000

# Kill any existing processes
get_ipython().system(f"pkill -f 'uvicorn.*{BACKEND_PORT}' 2>/dev/null || true")
time.sleep(1)

# Start backend in background
env = os.environ.copy()
env["PATH"] = f"{VENV}/bin:{env.get('PATH', '')}"
env["MEV_STRUCTURE_PROVIDER"] = "esmfold"
env["PYTHONPATH"] = WORKDIR

backend_proc = subprocess.Popen(
    [f"{VENV}/bin/python", "-m", "uvicorn", "app.main:app",
     "--host", "0.0.0.0", "--port", str(BACKEND_PORT), "--reload"],
    cwd=os.path.join(WORKDIR, "backend"),
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)

# Wait for backend to start
time.sleep(5)
if backend_proc.poll() is None:
    print(f"Backend started (PID: {backend_proc.pid})")
else:
    out = backend_proc.stdout.read().decode()
    print(f"Backend failed to start:", out[-500:])

# Start ngrok tunnel
try:
    from pyngrok import ngrok
    tunnel = ngrok.bind(BACKEND_PORT)
    public_url = tunnel.public_url
    print(f"\nngrok tunnel: {public_url}")
    print(f"Set this as NEXT_PUBLIC_API_URL in Vercel!")
except Exception as e:
    print(f"ngrok failed: {e}")
    print(f"Backend is running at http://localhost:{BACKEND_PORT}")

In [ ]:
#@title 11. Run the full pipeline (50 steps)
import requests, time, json

BASE = f"http://localhost:{BACKEND_PORT}"

# Create job
job_resp = requests.post(f"{BASE}/api/jobs", json={
    "taxonId": "99287",  # Streptococcus agalactiae
    "pathogenName": "Streptococcus agalactiae",
    "realTools": True,
})
job = job_resp.json()
job_id = job["id"]
print(f"Created job: {job_id}")

# Start pipeline
start_resp = requests.post(f"{BASE}/api/jobs/{job_id}/start")
print(f"Pipeline started: {start_resp.status_code}")

# Poll until complete
while True:
    status_resp = requests.get(f"{BASE}/api/jobs/{job_id}")
    status = status_resp.json()
    current_phase = status.get("currentPhase", 0)
    current_step = status.get("currentStep", "")
    pipeline_status = status.get("status", "unknown")
    
    if pipeline_status in ("completed", "failed", "error"):
        print(f"\nPipeline {pipeline_status}!")
        break
    
    print(f"Phase {current_phase}, Step {current_step}...", end="\r")
    time.sleep(10)

# Print final results
print(f"\n\n=== FINAL RESULTS ===")
print(json.dumps(status.get("funnel", {}), indent=2))

# Print MEV sequence if available
phases = status.get("phases", [])
for phase in phases:
    for step in phase.get("steps", []):
        if step.get("id") == "9-2" and step.get("result"):
            result = step["result"]
            mev_seq = result.get("sequence", "")
            mev_len = result.get("mev_length", 0)
            print(f"\nMEV Construct: {mev_len} aa")
            print(f"Sequence: {mev_seq[:100]}...")

In [ ]:
#@title 12. Check ESMFold structure results
import requests, json

BASE = f"http://localhost:{BACKEND_PORT}"
status_resp = requests.get(f"{BASE}/api/jobs/{job_id}")
status = status_resp.json()

# Find step 11-2 (structure prediction)
for phase in status.get("phases", []):
    for step in phase.get("steps", []):
        if step.get("id") == "11-2":
            print(f"Step 11-2 status: {step.get('status')}")
            result = step.get("result", {})
            print(f"Provider: {result.get('provider')}")
            print(f"Method: {result.get('method')}")
            print(f"Source: {result.get('source')}")
            seq_val = result.get("sequenceIdentityValidation", {})
            print(f"Sequence validation: identity={seq_val.get('identityPercent')}%, coverage={seq_val.get('coveragePercent')}%")